In [ ]:
from pathlib import Path
import sys

# Make this package's scripts/ importable (exp_info, regions, data_utils, ...).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "exp_info.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

In [ ]:
# Parameters: directories and settings for this run. Edit them for another
# machine or experiment, or override with papermill (-p NAME VALUE).
TOP_DIR  = "/lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble"  # input
DIAG_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3LE_paper/diag_data"  # output diagnostic data

In [6]:
import os, re, glob, json
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd

In [9]:
class ArcticAAMonthlyMetrics:
    """
    Boxplot-ready Arctic Amplification (AA) metrics by calendar month + ANN.

    Expects region-mean monthly NetCDF with dims (region,time) and ENS inferred from filename:
      - ERA5.en00.region_means.YYYYMM-YYYYMM.nc  (or ERA5.region_means.*.nc)
      - v3.LR.historical.enXX.region_means.185001-205012.nc

    Variables: TS(region,time), with regions named 'global' and 'Arctic' (case-insensitive).

    Public API:
      - build_payload(...): returns dict with per-month per-member AA and obs AA
      - save_netcdf_metrics(payload, path): writes compact .nc for plotting
      - run_and_save(...): convenience = build_payload + save_netcdf_metrics
    """

    # ---------------- config ----------------
    def __init__(
        self,
        *,
        data_dir,
        var="TS",
        pclimo=(1981, 2010),
        ptargt=(1979, 2018),
        trend_window=30,                 # years
        trend_anchor="center",           # 'center' | 'start' | 'end'
        denom_eps=1e-3,                  # K/decade guard for denominator
        verbose=True,
    ):
        self.data_dir = data_dir
        self.var = var
        self.pclimo = tuple(pclimo)
        self.ptargt = tuple(ptargt)
        self.trend_window = int(trend_window)
        self.trend_anchor = str(trend_anchor).lower()
        assert self.trend_anchor in ("center", "start", "end")
        self.denom_eps = float(denom_eps)
        self.verbose = verbose

        self._month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

    # ---------------- tiny helpers ----------------
    @staticmethod
    def _ens_index(path):
        m = re.search(r"\.en(\d+)\.", os.path.basename(path))
        return int(m.group(1)) if m else -1

    @staticmethod
    def _to_decimal_years(time_coord):
        try:
            ti = pd.to_datetime(time_coord.values)
            return ti.year + (ti.dayofyear - 0.5) / 365.25
        except Exception:
            return np.array(
                [t.year + (t.timetuple().tm_yday - 0.5) / 365.25 for t in time_coord.values],
                dtype=float,
            )

    @staticmethod
    def _window_counts(years, win_years):
        years = np.asarray(years, float)
        if years.size < 2:
            return 3
        dt = np.nanmedian(np.diff(years))
        return max(3, int(round(win_years / dt)))

    @staticmethod
    def _running_linear_trend_full(y, x, win_years, anchor="center"):
        """OLS slope (K/yr) only where the *full* window is available."""
        y = np.asarray(y, float)
        x = np.asarray(x, float)
        n = y.size
        out = np.full(n, np.nan)
        if n < 3:
            return out
        win_n = ArcticAAMonthlyMetrics._window_counts(x, win_years)
        anchor = str(anchor).lower()

        if anchor == "center":
            half = win_n // 2
            for i in range(half, n - (win_n - half) + 1):
                i0 = i - half
                i1 = i + half if (win_n % 2 == 0) else i + half + 1
                xx = x[i0:i1]; yy = y[i0:i1]
                if np.isfinite(xx).sum() != win_n or np.isfinite(yy).sum() != win_n:
                    continue
                X = np.vstack([xx, np.ones(win_n)]).T
                beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
                out[i] = beta[0]
            return out

        if anchor == "start":
            last = n - win_n
            if last < 0: return out
            for i in range(0, last + 1):
                xx = x[i:i+win_n]; yy = y[i:i+win_n]
                if np.isfinite(xx).sum() != win_n or np.isfinite(yy).sum() != win_n:
                    continue
                X = np.vstack([xx, np.ones(win_n)]).T
                beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
                out[i] = beta[0]
            return out

        # 'end'
        first = win_n - 1
        if first >= n: return out
        for i in range(first, n):
            xx = x[i-win_n+1:i+1]; yy = y[i-win_n+1:i+1]
            if np.isfinite(xx).sum() != win_n or np.isfinite(yy).sum() != win_n:
                continue
            X = np.vstack([xx, np.ones(win_n)]).T
            beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
            out[i] = beta[0]
        return out

    @staticmethod
    def _label_years_for_anchor(years, win_years, anchor):
        years = np.asarray(years, float)
        n = years.size
        lab = np.full(n, np.nan)
        win_n = ArcticAAMonthlyMetrics._window_counts(years, win_years)
        anchor = str(anchor).lower()

        if anchor == "center":
            half = win_n // 2
            for i in range(half, n - (win_n - half) + 1):
                i0 = i - half
                i1 = i + half if (win_n % 2 == 0) else i + half + 1
                L, R = years[i0], years[i1 - 1]
                lab[i] = 0.5 * (L + R)
        elif anchor == "start":
            last = n - win_n
            if last >= 0:
                for i in range(0, last + 1):
                    lab[i] = years[i]
        else:  # 'end'
            first = win_n - 1
            if first < n:
                for i in range(first, n):
                    lab[i] = years[i]
        return lab

    # ---------------- IO ----------------
    def _discover_paths(self, group, period, nens):
        y0m, y1m = f"{period[0]}01", f"{period[1]}12"
        if nens == 1:
            p1 = os.path.join(self.data_dir, f"{group}.en00.region_means.{y0m}-{y1m}.nc")
            p2 = os.path.join(self.data_dir, f"{group}.region_means.{y0m}-{y1m}.nc")
            paths = [p for p in (p1, p2) if os.path.exists(p)]
        else:
            pat = os.path.join(self.data_dir, f"{group}.en??.region_means.{y0m}-{y1m}.nc")
            paths = sorted(glob.glob(pat), key=self._ens_index)
        if self.verbose:
            print(f"[discover] {group} nens={nens} -> {len(paths)} file(s)")
        if not paths:
            raise FileNotFoundError(f"No files for group={group}, period={period}, nens={nens}")
        return paths

    def _open_concat(self, paths):
        dsets = []
        for p in paths:
            ds = xr.open_dataset(p, decode_times=True, use_cftime=True)
            if "region" not in ds.dims:
                raise ValueError(f"'region' dimension missing in {p}")
            regs = [str(r) for r in ds["region"].values]
            def pick(name):
                return next((r for r in regs if r.lower() == name.lower()), None)
            g = pick("global"); a = pick("Arctic")
            if g is None or a is None:
                raise ValueError(f"regions ['global','Arctic'] missing in {p}")
            ds = ds[[self.var]].sel(region=[g, a])
            ens = f"en{str(self._ens_index(p)).zfill(2)}"
            dsets.append(ds.expand_dims(ens=[ens]))
        if len(dsets) == 1 and "ens" not in dsets[0].dims:
            dsets[0] = dsets[0].expand_dims(ens=["en00"])
        return xr.concat(dsets, dim="ens")

    # ---------------- cores ----------------
    def _monthly_anomaly(self, da):
        """Remove monthly climatology over fixed PCLIMO."""
        try:
            yr = da["time"].dt.year
        except Exception:
            yr = xr.DataArray(np.floor(self._to_decimal_years(da["time"])).astype(int), dims=("time",))
        base = da.sel(time=(yr >= self.pclimo[0]) & (yr <= self.pclimo[1]))
        clim = base.groupby("time.month").mean("time")
        return da.groupby("time.month") - clim

    def _select_month(self, da, month):
        try:
            msel = da["time"].dt.month == int(month)
        except Exception:
            msel = xr.DataArray(
                np.array([getattr(t, "month", np.nan) == int(month) for t in da["time"].values]),
                dims=("time",),
            )
        return da.sel(time=msel)

    def _compute_month(self, paths, month):
        """Return AA(ens,time_year) for one calendar month."""
        ds = self._open_concat(paths)
        DT = self._monthly_anomaly(ds[self.var])
        glb_all = DT.sel(region=DT.region.values[0])
        arc_all = DT.sel(region=DT.region.values[1])

        glb = self._select_month(glb_all, month)
        arc = self._select_month(arc_all, month)

        yrs_all = self._to_decimal_years(glb["time"])
        sel = (yrs_all >= self.ptargt[0]) & (yrs_all <= self.ptargt[1])
        glb = glb.sel(time=sel); arc = arc.sel(time=sel); yrs = yrs_all[sel]

        bet_g, bet_a = [], []
        for i in range(glb.sizes["ens"]):
            yg = glb.isel(ens=i).values
            ya = arc.isel(ens=i).values
            bg = self._running_linear_trend_full(yg, yrs, self.trend_window, anchor=self.trend_anchor) * 10.0
            ba = self._running_linear_trend_full(ya, yrs, self.trend_window, anchor=self.trend_anchor) * 10.0
            bet_g.append(bg); bet_a.append(ba)

        bet_g = xr.DataArray(np.vstack(bet_g), dims=("ens","time"), coords={"ens": glb["ens"], "time": glb["time"]})
        bet_a = xr.DataArray(np.vstack(bet_a), dims=("ens","time"), coords={"ens": arc["ens"], "time": arc["time"]})

        valid = np.isfinite(bet_g).any("ens") & np.isfinite(bet_a).any("ens")
        lab_year = self._label_years_for_anchor(yrs, self.trend_window, self.trend_anchor)
        lab_ok = np.isfinite(lab_year) & valid.values

        yrs_full = np.arange(self.ptargt[0], self.ptargt[1] + 1, dtype=int)
        yrs_lbl = np.round(lab_year[lab_ok]).astype(int)

        def _reindex(da):
            da = da.sel(time=lab_ok)
            da = da.assign_coords(time_year=("time", yrs_lbl))
            return da.swap_dims({"time":"time_year"}).reindex(time_year=yrs_full)

        bet_g = _reindex(bet_g)
        bet_a = _reindex(bet_a)
        denom = xr.where(np.abs(bet_g) < self.denom_eps, np.nan, bet_g)
        aa = bet_a / denom
        return aa  # (ens,time_year)

    def _compute_ann(self, paths):
        """Annual means → AA(ens,time_year)."""
        ds = self._open_concat(paths)
        DT = self._monthly_anomaly(ds[self.var])
        ann = DT.resample(time="YS").mean(keep_attrs=True)
        glb = ann.sel(region=ann.region.values[0])
        arc = ann.sel(region=ann.region.values[1])

        yrs_all = self._to_decimal_years(glb["time"])
        sel = (yrs_all >= self.ptargt[0]) & (yrs_all <= self.ptargt[1])
        glb = glb.sel(time=sel); arc = arc.sel(time=sel); yrs = yrs_all[sel]

        bet_g, bet_a = [], []
        for i in range(glb.sizes["ens"]):
            yg = glb.isel(ens=i).values
            ya = arc.isel(ens=i).values
            bg = self._running_linear_trend_full(yg, yrs, self.trend_window, anchor=self.trend_anchor) * 10.0
            ba = self._running_linear_trend_full(ya, yrs, self.trend_window, anchor=self.trend_anchor) * 10.0
            bet_g.append(bg); bet_a.append(ba)

        bet_g = xr.DataArray(np.vstack(bet_g), dims=("ens","time"), coords={"ens": glb["ens"], "time": glb["time"]})
        bet_a = xr.DataArray(np.vstack(bet_a), dims=("ens","time"), coords={"ens": arc["ens"], "time": arc["time"]})

        valid = np.isfinite(bet_g).any("ens") & np.isfinite(bet_a).any("ens")
        lab_year = self._label_years_for_anchor(yrs, self.trend_window, self.trend_anchor)
        lab_ok = np.isfinite(lab_year) & valid.values

        yrs_full = np.arange(self.ptargt[0], self.ptargt[1] + 1, dtype=int)
        yrs_lbl = np.round(lab_year[lab_ok]).astype(int)

        def _reindex(da):
            da = da.sel(time=lab_ok)
            da = da.assign_coords(time_year=("time", yrs_lbl))
            return da.swap_dims({"time":"time_year"}).reindex(time_year=yrs_full)

        bet_g = _reindex(bet_g)
        bet_a = _reindex(bet_a)
        denom = xr.where(np.abs(bet_g) < self.denom_eps, np.nan, bet_g)
        return bet_a / denom  # (ens,time_year)

    # ---------------- API: payload + NetCDF ----------------
    def build_payload(
        self,
        *,
        model_group, model_period, model_nens,
        obs_group,   obs_period,   obs_nens=1,
        include_ann=True,
        aa_gate_for_pct=2.0,
    ):
        """Return dict with per-month model arrays, obs scalars, and AA≥gate percentages."""
        model_paths = self._discover_paths(model_group, model_period, model_nens)
        obs_paths   = self._discover_paths(obs_group,   obs_period,   obs_nens)

        months_num = list(range(1, 13))
        model_values = {}
        obs_value = {}
        frac_ge2_pct = {}

        # 1..12
        for m in months_num:
            aa_m_model = self._compute_month(model_paths, m)     # (ens,time_year)
            aa_m_obs   = self._compute_month(obs_paths,   m)     # (ens=1,time_year)

            valid_t = np.isfinite(aa_m_model).any("ens")
            aa_model_valid = aa_m_model.sel(time_year=valid_t)

            per_member = aa_model_valid.mean("time_year").values  # (nEns,)
            model_values[m] = np.asarray(per_member, float)

            valid_t_obs = np.isfinite(aa_m_obs).any("ens")
            obs_scalar = aa_m_obs.sel(time_year=valid_t_obs).mean("time_year").values
            obs_value[m] = np.asarray(obs_scalar).item()          # Python float

            frac_time = (aa_model_valid >= aa_gate_for_pct).mean("time_year")
            frac_ge2 = float(frac_time.mean("ens").values) * 100.0
            frac_ge2_pct[m] = frac_ge2

        # ANN
        if include_ann:
            aaA_model = self._compute_ann(model_paths)
            aaA_obs   = self._compute_ann(obs_paths)

            valid_tA = np.isfinite(aaA_model).any("ens")
            aaA_model_valid = aaA_model.sel(time_year=valid_tA)
            per_memberA = aaA_model_valid.mean("time_year").values
            model_values["ANN"] = np.asarray(per_memberA, float)

            valid_tA_obs = np.isfinite(aaA_obs).any("ens")
            obs_scalarA = aaA_obs.sel(time_year=valid_tA_obs).mean("time_year").values
            obs_value["ANN"] = np.asarray(obs_scalarA).item()

            frac_timeA = (aaA_model_valid >= aa_gate_for_pct).mean("time_year")
            frac_ge2A = float(frac_timeA.mean("ens").values) * 100.0
            frac_ge2_pct["ANN"] = frac_ge2A

        return {
            "months": self._month_labels + (["Ann"] if include_ann else []),
            "model_values": model_values,         # dict: {1..12, "ANN"} -> (nEns,) array
            "obs_value": obs_value,               # dict: {1..12, "ANN"} -> float
            "frac_AA_ge2_pct": frac_ge2_pct,      # dict: {1..12, "ANN"} -> float (%)
            "gate": float(aa_gate_for_pct),
            "var": self.var,
            "pclimo": self.pclimo,
            "ptargt": self.ptargt,
            "trend_window_years": self.trend_window,
            "trend_anchor": self.trend_anchor,
        }

    def save_netcdf_metrics(self, payload, path):
        """
        Save a compact NetCDF with:
          aa_mean(ens, month), obs_aa(month), frac_AA_ge2(month)
        Coords:
          month_index: [1..12, 13(Ann)] ; month_name: ["Jan"..,"Dec","Ann"]
        """
        # Order months + ANN
        keys = list(range(1, 13)) + (["ANN"] if "ANN" in payload["model_values"] else [])
        names = self._month_labels + (["Ann"] if "ANN" in keys else [])

        # Build 2D array (nEns, nMon)
        ens_arrays = [np.asarray(payload["model_values"][k], float) for k in keys]
        # assume constant ensemble size; pad just in case
        n_ens = max(len(a) for a in ens_arrays)
        def _pad(a): return np.pad(a, (0, n_ens - len(a)), constant_values=np.nan)
        data = np.vstack([_pad(a) for a in ens_arrays]).T  # (n_ens, n_mon)

        obs_vals  = np.array([payload["obs_value"][k] for k in keys], dtype=float)
        frac_vals = np.array([payload["frac_AA_ge2_pct"][k] for k in keys], dtype=float)

        # month_index: 1..12 (+13 for Ann), month_name labels
        month_index = list(range(1, 13)) + ([13] if len(keys) == 13 else [])
        month_name = names

        ds = xr.Dataset(
            data_vars=dict(
                aa_mean=(["ens", "month"], data),
                obs_aa=(["month"], obs_vals),
                frac_AA_ge2=(["month"], frac_vals),
            ),
            coords=dict(
                ens=np.arange(n_ens, dtype=np.int32),
                month=("month", np.array(month_index, dtype=np.int16)),
                month_name=("month", np.array(month_name, dtype="U3")),
            ),
            attrs=dict(
                var=payload.get("var"),
                pclimo=f"{payload['pclimo'][0]}-{payload['pclimo'][1]}",
                ptargt=f"{payload['ptargt'][0]}-{payload['ptargt'][1]}",
                trend_window_years=int(payload["trend_window_years"]),
                trend_anchor=str(payload["trend_anchor"]),
                gate=float(payload["gate"]),
                note="AA_mean is per-member monthly mean within PTARGT; obs_aa from ERA5; frac_AA_ge2 in percent.",
            ),
        )

        enc = {
            "aa_mean":     {"zlib": True, "complevel": 2, "dtype": "float32"},
            "obs_aa":      {"zlib": True, "complevel": 2, "dtype": "float32"},
            "frac_AA_ge2": {"zlib": True, "complevel": 2, "dtype": "float32"},
            "ens":         {"dtype": "int32"},
            "month":       {"dtype": "int16"},
            "month_name":  {"zlib": False},
        }
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        ds.load().to_netcdf(path, encoding=enc)
        if self.verbose:
            print(f"[OK] wrote {path}")

    def run_and_save(
        self,
        *,
        model_group, model_period, model_nens,
        obs_group,   obs_period,   obs_nens=1,
        include_ann=True,
        aa_gate_for_pct=2.0,
        out_nc_path=None,
    ):
        """Convenience: compute → write NetCDF → return (payload, nc_path)."""
        payload = self.build_payload(
            model_group=model_group, model_period=model_period, model_nens=model_nens,
            obs_group=obs_group,     obs_period=obs_period,     obs_nens=obs_nens,
            include_ann=include_ann,
            aa_gate_for_pct=aa_gate_for_pct,
        )
        if out_nc_path:
            self.save_netcdf_metrics(payload, out_nc_path)
        return payload, out_nc_path


In [10]:
if __name__ == "__main__":
    # ----------------------------------------------------------------------
    # Configuration
    # ----------------------------------------------------------------------
    DATA_DIR  = f"{TOP_DIR}/data/regmn"
    OUT_DIR   = DIAG_DIR
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    # ---- Model / Obs pairing ----
    MODEL_GROUP   = "v3.LR.historical"
    MODEL_PERIOD  = (1850, 2050)
    MODEL_NENS    = 25

    OBS_GROUP     = "ERA5"
    OBS_PERIOD    = (1979, 2018)
    OBS_NENS      = 1

    # ---- AA settings ----
    PTARGT        = (1979, 2018)
    PCLIMO        = (1981, 2010)
    VAR           = "TS"
    TREND_WINDOW  = 30
    GATE          = 2.0
    VERBOSE       = True

    # ----------------------------------------------------------------------
    # Run monthly AA metrics processor (writes a single .nc)
    # ----------------------------------------------------------------------
    engine = ArcticAAMonthlyMetrics(
        data_dir=DATA_DIR,
        var=VAR,
        pclimo=PCLIMO,
        ptargt=PTARGT,
        trend_window=TREND_WINDOW,
        trend_anchor="center",
        verbose=VERBOSE,
    )

    out_nc = f"{OUT_DIR}/AA_monthly_metrics_1979-2018.nc"
    payload, _ = engine.run_and_save(
        model_group=MODEL_GROUP, model_period=MODEL_PERIOD, model_nens=MODEL_NENS,
        obs_group=OBS_GROUP,     obs_period=OBS_PERIOD,     obs_nens=OBS_NENS,
        include_ann=True,
        aa_gate_for_pct=GATE,
        out_nc_path=out_nc,
    )

    # ----------------------------------------------------------------------
    # Save a small manifest (optional)
    # ----------------------------------------------------------------------
    manifest = {
        "model_group": MODEL_GROUP,
        "model_period": MODEL_PERIOD,
        "model_nens": MODEL_NENS,
        "obs_group": OBS_GROUP,
        "obs_period": OBS_PERIOD,
        "obs_nens": OBS_NENS,
        "ptargt": PTARGT,
        "pclimo": PCLIMO,
        "var": VAR,
        "trend_window": TREND_WINDOW,
        "trend_anchor": "center",
        "gate": GATE,
        "netcdf": str(Path(out_nc)),
    }
    man_path = Path(OUT_DIR, "AA_monthly_metrics_manifest.json")
    with open(man_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"[OK] wrote {man_path}")

    # ----------------------------------------------------------------------
    # Quick check printout
    # ----------------------------------------------------------------------
    print("\n[SUMMARY] Arctic Amplification monthly diagnostics (1979–2018):")
    for k in payload["obs_value"]:
        v   = payload["obs_value"][k]
        pct = payload["frac_AA_ge2_pct"][k]
        print(f"  {str(k):>3}:  Obs={v:4.2f},  P(AA>2)={pct:5.1f}%")

    print(f"\n[OK] NetCDF saved to: {out_nc}")


[discover] v3.LR.historical nens=25 -> 25 file(s)
[discover] ERA5 nens=1 -> 1 file(s)
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/figure_data/AA_monthly_metrics_1979-2018.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/figure_data/AA_monthly_metrics_manifest.json

[SUMMARY] Arctic Amplification monthly diagnostics (1979–2018):
    1:  Obs=2.14,  P(AA>2)= 99.6%
    2:  Obs=2.43,  P(AA>2)= 94.4%
    3:  Obs=3.71,  P(AA>2)= 85.6%
    4:  Obs=2.75,  P(AA>2)= 81.2%
    5:  Obs=2.41,  P(AA>2)= 28.8%
    6:  Obs=2.15,  P(AA>2)= 19.2%
    7:  Obs=2.21,  P(AA>2)= 37.6%
    8:  Obs=2.90,  P(AA>2)= 58.8%
    9:  Obs=3.33,  P(AA>2)= 76.8%
   10:  Obs=3.80,  P(AA>2)= 98.8%
   11:  Obs=3.78,  P(AA>2)=100.0%
   12:  Obs=3.52,  P(AA>2)=100.0%
  ANN:  Obs=2.99,  P(AA>2)= 96.4%

[OK] NetCDF saved to: /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/large_ensemble/figure_data/AA_monthly_metrics_1979-2018.nc
